In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Nhavi(Indapur)_Report_29_7_2026-30_7_2026.csv to Nhavi(Indapur)_Report_29_7_2026-30_7_2026.csv


In [ ]:
# import sqlite and create uniform timestamp
import sqlite3
import pandas as pd
df = pd.read_csv('Nhavi(Indapur)_Report_29_7_2026-30_7_2026.csv')
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
print(df[['date', 'time', 'datetime']].head())
print(df['datetime'].dtype)



         date      time            datetime
0  2026-07-29  00:00:00 2026-07-29 00:00:00
1  2026-07-29  00:02:00 2026-07-29 00:02:00
2  2026-07-29  00:04:00 2026-07-29 00:04:00
3  2026-07-29  00:06:00 2026-07-29 00:06:00
4  2026-07-29  00:08:00 2026-07-29 00:08:00
datetime64[ns]


In [ ]:
# create sqlite database and connect it to load data
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
        print("Connection to SQLite DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")
    return connection

conn = create_connection('spectra.db')
df.to_sql('sensor_data', conn, if_exists='replace', index=False)

Connection to SQLite DB successful


1071

In [ ]:
#take data from sqlite
query = "SELECT * FROM sensor_data ORDER BY datetime"
df = pd.read_sql(query, conn)
df['datetime'] = pd.to_datetime(df['datetime'])
df.head()

,date,time,Faultcount_,Faultcount,DischargePumpStatus,ChillingSwitchStatus,ChargingFaultStatusFlag_,FaultLP_,FaultHP_,CompressorRunHoursMinutes_,...,AuxTemperature,TSSSetpoint,ACVoltage_,ACVoltage,AppTemperature,TSSTemperature,BatteryVoltage,CompressorCurrent,CompressorCurrent_,datetime
0,2026-07-29,00:00:00,0,29952,0,1,0,0,0,0,...,-13,-60,0,238,3.2,-3.5,26,7.9,0,2026-07-29 00:00:00
1,2026-07-29,00:02:00,0,29952,0,1,0,0,0,0,...,-13,-60,0,238,3.4,-3.6,26,7.9,0,2026-07-29 00:02:00
2,2026-07-29,00:04:00,0,29952,0,1,0,0,0,0,...,-13,-60,0,238,3.5,-3.7,26,7.9,0,2026-07-29 00:04:00
3,2026-07-29,00:06:00,0,29952,0,1,0,0,0,0,...,-13,-60,0,239,3.7,-3.9,26,7.8,0,2026-07-29 00:06:00
4,2026-07-29,00:08:00,0,29952,0,1,0,0,0,0,...,-13,-60,0,240,3.8,-4.0,26,7.8,0,2026-07-29 00:08:00


In [ ]:
#take the sensor columns and add averages
sensor_cols = ['ACVoltage', 'TSSTemperature', 'BatteryVoltage', 'CompressorCurrent']

for col in sensor_cols:
    df[col + '_avg'] = df[col].rolling(15).mean()  # average of last 15 readings (~30 min)

df = df.dropna().reset_index(drop=True)
df.head()

,date,time,Faultcount_,Faultcount,DischargePumpStatus,ChillingSwitchStatus,ChargingFaultStatusFlag_,FaultLP_,FaultHP_,CompressorRunHoursMinutes_,...,AppTemperature,TSSTemperature,BatteryVoltage,CompressorCurrent,CompressorCurrent_,datetime,ACVoltage_avg,TSSTemperature_avg,BatteryVoltage_avg,CompressorCurrent_avg
0,2026-07-29,00:28:00,0,29952,0,1,0,0,0,0,...,3.9,-5.0,26,7.7,0,2026-07-29 00:28:00,238.733333,-4.266667,26.0,7.793333
1,2026-07-29,00:30:00,0,29952,0,1,0,0,0,0,...,3.4,-5.1,26,7.8,0,2026-07-29 00:30:00,238.866667,-4.373333,26.0,7.786667
2,2026-07-29,00:32:00,0,29952,0,1,0,0,0,0,...,3.6,-5.1,26,7.7,0,2026-07-29 00:32:00,239.000000,-4.473333,26.0,7.773333
3,2026-07-29,00:34:00,0,29952,0,1,0,0,0,0,...,3.6,-5.3,26,7.7,0,2026-07-29 00:34:00,239.133333,-4.580000,26.0,7.760000
4,2026-07-29,00:36:00,0,29952,0,1,0,0,0,0,...,3.8,-5.3,26,7.7,0,2026-07-29 00:36:00,239.200000,-4.673333,26.0,7.753333


In [ ]:
# anomaly detection model
from sklearn.ensemble import IsolationForest

feature_cols = sensor_cols + [c + '_avg' for c in sensor_cols]
X = df[feature_cols]

model = IsolationForest(contamination=0.03, random_state=42)
model.fit(X)

df['is_anomaly'] = model.predict(X) == -1
print(f"Flagged {df['is_anomaly'].sum()} anomalies out of {len(df)} readings")


Flagged 32 anomalies out of 1057 readings


In [ ]:
df[df['is_anomaly']][['datetime'] + sensor_cols].head(10)

,datetime,ACVoltage,TSSTemperature,BatteryVoltage,CompressorCurrent
11,2026-07-29 00:50:00,240,-6.0,26,0.0
13,2026-07-29 00:54:00,236,-6.2,26,0.0
251,2026-07-29 08:50:00,232,-3.7,26,7.2
252,2026-07-29 08:52:00,234,-3.6,26,7.9
256,2026-07-29 09:00:00,233,-3.5,26,8.2
287,2026-07-29 10:02:00,235,-1.4,26,0.0
338,2026-07-29 11:44:00,0,0.5,25,0.0
339,2026-07-29 11:46:00,242,0.5,26,0.0
340,2026-07-29 11:48:00,242,0.5,26,0.0
341,2026-07-29 11:50:00,241,0.5,26,8.9


In [ ]:
#export to powerbi
df[['datetime'] + sensor_cols + ['is_anomaly']].to_csv('nhavi_anomalies.csv', index=False)
print("Saved!")

Saved!


In [ ]:

# download it
from google.colab import files
files.download('nhavi_anomalies.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>